# Huấn luyện và đánh giá mô hình Gaming

Dự đoán **3DMark Time Spy Overall** từ **GpuScore** (PassMark G3D Mark)
và **CpuMultiScore** (PassMark CPU Mark) bằng hồi quy log–log:

$$\hat y = A\times GpuScore^b\times CpuMultiScore^c,\quad b,c\geq0.$$

**Cách chạy trên Colab:** chọn **Runtime → Run all**, tải lên đúng một tệp
Excel benchmark khi được yêu cầu, chờ các ô chạy xong và nhận gói ZIP ở cuối.
Excel cần có hai sheet `Benchmark_Gaming` và `ReferenceBenchmark`.
Với dữ liệu dự án hiện tại, dùng tệp `data/raw/Benchmark final.xlsx`.

Notebook thực hiện theo thứ tự: đọc dữ liệu → xử lý → chia dữ liệu và học
mô hình đánh giá → tính MAE/RMSE/R² → học mô hình triển khai trên toàn bộ
dữ liệu → xuất kết quả.

`eval_model` chỉ học tập train; `final_model` học toàn bộ dữ liệu hợp lệ.
Điểm benchmark dự đoán không phải FPS của từng trò chơi.

## 1. Đọc dữ liệu từ Excel

Tải lên file Excel đã chỉnh sửa và đọc hai sheet:

- `Benchmark_Gaming`: các lượt chạy benchmark, cấu hình máy và điểm Time Spy.
- `ReferenceBenchmark`: điểm PassMark dùng làm đầu vào cho mô hình.

Bước này chỉ đọc dữ liệu, kiểm tra tên sheet và thống kê trạng thái duyệt. Chưa loại dòng, ghép điểm hoặc huấn luyện mô hình.

Mã SHA256 được lưu để nhận diện chính xác phiên bản file Excel đã sử dụng.

In [ ]:
import io
import hashlib
import pandas as pd

from google.colab import files
from IPython.display import display

# Tải lên đúng một file Excel.
uploaded = files.upload()

excel_names = [
    name for name in uploaded
    if name.lower().endswith(".xlsx")
]

if len(excel_names) != 1:
    raise ValueError("Hãy tải lên đúng một file Excel .xlsx.")

excel_name = excel_names[0]
excel_bytes = uploaded[excel_name]
excel_sha256 = hashlib.sha256(excel_bytes).hexdigest()

# Đọc và kiểm tra hai sheet cần thiết.
sheets = pd.read_excel(io.BytesIO(excel_bytes), sheet_name=None)

required_sheets = ["Benchmark_Gaming", "ReferenceBenchmark"]
missing_sheets = [
    name for name in required_sheets if name not in sheets
]

if missing_sheets:
    raise ValueError(f"File thiếu các sheet: {missing_sheets}")

gaming_raw = sheets["Benchmark_Gaming"].copy()
reference_raw = sheets["ReferenceBenchmark"].copy()

for frame in (gaming_raw, reference_raw):
    frame.columns = frame.columns.str.strip()

if "ReviewStatus" not in gaming_raw.columns:
    raise ValueError("Sheet Benchmark_Gaming thiếu cột ReviewStatus.")

# Chỉ thống kê, chưa lọc dữ liệu.
print("File:", excel_name)
print("SHA256:", excel_sha256)
print("Số dòng Gaming:", len(gaming_raw))
print("Số dòng Reference:", len(reference_raw))

print("\nTrạng thái duyệt của dữ liệu Gaming:")
display(
    gaming_raw["ReviewStatus"]
    .value_counts(dropna=False)
    .rename_axis("ReviewStatus")
    .to_frame("Số dòng")
)

## 2. Chuẩn bị dữ liệu đầu vào

Chỉ sử dụng các lượt benchmark có trạng thái `ACCEPT`, bài kiểm tra `Time Spy` và chỉ số `Overall`. Các dòng còn lại được lưu riêng để theo dõi.

Ghép hai điểm tham chiếu từ sheet `ReferenceBenchmark`:

- `CpuMultiScore`: CPU Mark của CPU.
- `GpuScore`: G3D Mark của GPU.

Tên linh kiện được chuẩn hóa để xử lý khác biệt cách viết như khoảng trắng, dấu gạch nối, từ “Processor” và tiền tố “NVIDIA”. Mã linh kiện, hậu tố và dung lượng VRAM được giữ nguyên; không sử dụng ghép gần đúng.

Dừng xử lý nếu mã lượt chạy hoặc nguồn benchmark bị thiếu/trùng, điểm tham chiếu không xác định duy nhất, hoặc điểm đầu vào/điểm đích không phải số hữu hạn lớn hơn 0. Không thay dữ liệu thiếu bằng 0.

RAM không nằm trong công thức của mô hình này. Bước chuẩn bị vẫn giữ các cột gốc để truy vết dữ liệu.

In [ ]:
import re
import numpy as np

# Làm việc trên bản sao, giữ nguyên dữ liệu đã đọc từ Excel.
gaming = gaming_raw.copy()
reference = reference_raw.copy()

required_gaming = [
    "RunId", "CpuModel", "GpuModel", "TestName",
    "Metric", "Score", "SourceUrl", "ReviewStatus"
]
required_reference = [
    "ReferenceId", "ComponentType", "ModelName",
    "TestName", "Metric", "RawScore", "ReviewStatus"
]

for label, frame, columns in [
    ("Gaming", gaming, required_gaming),
    ("Reference", reference, required_reference)
]:
    missing = [column for column in columns if column not in frame.columns]
    if missing:
        raise ValueError(f"{label} thiếu cột: {missing}")

    # Bỏ khoảng trắng thừa, giữ ô thiếu ở dạng thiếu.
    for column in frame.select_dtypes(include=["object", "string"]).columns:
        frame[column] = frame[column].astype("string").str.strip()


# 1. Lọc đúng loại benchmark và trạng thái duyệt.
eligible = (
    gaming["ReviewStatus"].str.upper().eq("ACCEPT")
    & gaming["TestName"].eq("Time Spy")
    & gaming["Metric"].eq("Overall")
).fillna(False)

excluded_rows = gaming.loc[~eligible].copy()
data = gaming.loc[eligible].copy()

if data.empty:
    raise ValueError("Không có dòng Gaming hợp lệ để huấn luyện.")

for column in ["RunId", "SourceUrl"]:
    invalid = (
        data[column].isna()
        | data[column].eq("")
        | data[column].duplicated(keep=False)
    )
    if invalid.any():
        display(data.loc[invalid, ["RunId", "SourceUrl"]])
        raise ValueError(f"{column} bị thiếu hoặc trùng; cần kiểm tra.")


# 2. Chuẩn hóa tên nhưng giữ mã CPU/GPU và dung lượng VRAM.
def component_key(value):
    if pd.isna(value):
        return ""

    text = str(value).lower().split("@")[0]
    text = re.sub(r"\b(processor|nvidia)\b", "", text)
    return re.sub(r"[^a-z0-9]", "", text)


data["CpuKey"] = data["CpuModel"].map(component_key)
data["GpuKey"] = data["GpuModel"].map(component_key)
reference["ComponentKey"] = reference["ModelName"].map(component_key)


# 3. Ghép điểm theo đúng linh kiện, bài kiểm tra và loại điểm.
mapping_specs = [
    ("CPU", "CPU Mark", "CpuKey", "CpuMultiScore", "CpuMultiRefId"),
    ("GPU", "G3D Mark", "GpuKey", "GpuScore", "GpuRefId")
]

for component, metric, key_column, score_column, id_column in mapping_specs:
    accepted_reference = (
        reference["ComponentType"].eq(component)
        & reference["TestName"].eq(f"PassMark {component}")
        & reference["Metric"].eq(metric)
        & reference["ReviewStatus"].str.upper().eq("ACCEPT")
    ).fillna(False)

    selected = reference.loc[accepted_reference].copy()

    if selected.empty:
        raise ValueError(f"Không tìm thấy điểm tham chiếu: {metric}.")

    bad_reference = (
        selected["ComponentKey"].eq("")
        | selected["ComponentKey"].duplicated(keep=False)
        | selected["ReferenceId"].isna()
        | selected["ReferenceId"].eq("")
        | selected["ReferenceId"].duplicated(keep=False)
    )
    if bad_reference.any():
        display(selected.loc[bad_reference])
        raise ValueError(f"Tham chiếu {metric} bị thiếu hoặc không duy nhất.")

    lookup = selected.set_index("ComponentKey")
    matched_ids = data[key_column].map(lookup["ReferenceId"])
    matched_scores = data[key_column].map(lookup["RawScore"])

    if matched_ids.isna().any():
        display(data.loc[
            matched_ids.isna(), ["RunId", "CpuModel", "GpuModel"]
        ])
        raise ValueError(f"Có linh kiện chưa ghép được điểm {metric}.")

    # Nếu Excel đã ghi mã tham chiếu, kiểm tra mã đó có khớp tên không.
    if id_column in data.columns:
        existing_ids = data[id_column].astype("string").str.strip()
        supplied = existing_ids.notna() & existing_ids.ne("")
        mismatch = (
            supplied & existing_ids.ne(matched_ids.astype("string"))
        ).fillna(False)

        if mismatch.any():
            display(data.loc[
                mismatch, ["RunId", "CpuModel", "GpuModel", id_column]
            ])
            raise ValueError(f"{id_column} mâu thuẫn với tên linh kiện.")

    data[id_column] = matched_ids
    data[score_column] = matched_scores


# 4. Log chỉ nhận điểm hữu hạn và lớn hơn 0.
numeric_columns = ["GpuScore", "CpuMultiScore", "Score"]

for column in numeric_columns:
    data[column] = pd.to_numeric(
        data[column], errors="coerce"
    ).astype(float)

invalid_numeric = (
    ~np.isfinite(data[numeric_columns])
    | data[numeric_columns].le(0)
).any(axis=1)

if invalid_numeric.any():
    display(data.loc[
        invalid_numeric, ["RunId"] + numeric_columns
    ])
    raise ValueError("Có điểm bị thiếu, không hữu hạn hoặc không lớn hơn 0.")

data = data.sort_values("RunId").reset_index(drop=True)


# 5. Hiển thị kết quả chuẩn bị dữ liệu.
print("Số dòng được dùng:", len(data))
print("Số dòng không dùng:", len(excluded_rows))
print("Số CPU:", data["CpuKey"].nunique())
print("Số GPU:", data["GpuKey"].nunique())

print("\nCác dòng không dùng:")
display(excluded_rows[["RunId", "ReviewStatus", "TestName", "Metric"]])

print("\nVí dụ dữ liệu đã ghép điểm:")
display(data[
    ["RunId", "CpuModel", "GpuModel",
     "CpuMultiScore", "GpuScore", "Score"]
].head())

print("\nKhoảng điểm trong dữ liệu:")
display(data[numeric_columns].agg(["min", "max"]))

## 3. Chia dữ liệu và huấn luyện mô hình đánh giá

### 3.1. Chia train và test theo nhóm CPU–GPU

Sử dụng `GroupShuffleSplit(test_size=0.20, random_state=42)`.
Khoảng 20% **số nhóm** được giữ làm test, không nhất thiết đúng 20% số dòng.
Các lượt đo cùng cặp CPU–GPU đã chuẩn hóa được giữ trong cùng một tập.

Nhóm này không phải mã máy vật lý. Tập test chứa những cặp CPU–GPU không có
trong train, nhưng từng model CPU hoặc GPU vẫn có thể đã xuất hiện trong train.
Giữ cố định cách chia; không thử nhiều seed để chọn kết quả test đẹp nhất.

### 3.2. Xây dựng và huấn luyện mô hình

$$\ln(\hat y)=a+b\ln(GpuScore)+c\ln(CpuMultiScore),\quad A=e^a.$$

`FunctionTransformer` lấy log đầu vào; `LinearRegression(positive=True)`
học hai hệ số không âm; `TransformedTargetRegressor` lấy log biến mục tiêu
khi học và dùng hàm mũ khi dự đoán. Hệ số chặn không bị ràng buộc không âm.

`eval_model` chỉ học từ tập train. Điểm thực tế của test không tham gia học hệ số.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor

eval_features = ["GpuScore", "CpuMultiScore"]
eval_groups = data["CpuKey"] + "__" + data["GpuKey"]
EVAL_RANDOM_STATE = 42
EVAL_TEST_GROUP_FRACTION = 0.20

if eval_groups.nunique() < 2:
    raise ValueError("Cần ít nhất hai nhóm CPU–GPU để chia train/test.")

eval_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=EVAL_TEST_GROUP_FRACTION,
    random_state=EVAL_RANDOM_STATE,
)
eval_train_idx, eval_test_idx = next(eval_splitter.split(
    data[eval_features], data["Score"], groups=eval_groups
))
eval_train = data.iloc[eval_train_idx].copy()
eval_test = data.iloc[eval_test_idx].copy()
eval_train_groups = set(eval_groups.iloc[eval_train_idx])
eval_test_groups = set(eval_groups.iloc[eval_test_idx])

assert eval_train_groups.isdisjoint(eval_test_groups)
assert len(eval_train) + len(eval_test) == len(data)
for label, frame in [("train", eval_train), ("test", eval_test)]:
    if len(frame) < 2 or frame["Score"].nunique() < 2:
        raise ValueError(f"Tập {label} không đủ mẫu hoặc biến thiên để tính R².")

print(f"Train: {len(eval_train)} mẫu, {len(eval_train_groups)} nhóm CPU–GPU")
print(f"Test: {len(eval_test)} mẫu, {len(eval_test_groups)} nhóm CPU–GPU")
print("Số nhóm trùng giữa train và test: 0")

eval_model = TransformedTargetRegressor(
    regressor=Pipeline([
        ("log_input", FunctionTransformer(np.log, validate=True)),
        ("linear", LinearRegression(positive=True)),
    ]),
    func=np.log,
    inverse_func=np.exp,
)
eval_model.fit(eval_train[eval_features], eval_train["Score"])
eval_linear = eval_model.regressor_.named_steps["linear"]
eval_coefficients = {
    "A": float(np.exp(eval_linear.intercept_)),
    "b": float(eval_linear.coef_[0]),
    "c": float(eval_linear.coef_[1]),
}
print("Hệ số của mô hình ĐÁNH GIÁ, chỉ học tập train:")
display(pd.DataFrame([eval_coefficients]))

## 4. Đánh giá trên tập train và tập test

Đưa các điểm CPU/GPU vào `eval_model`, rồi so sánh dự đoán với điểm Time Spy
Overall thực tế. Các chỉ số được tính trên thang điểm gốc:

- **MAE:** sai số tuyệt đối trung bình.
- **RMSE:** căn sai số bình phương trung bình, nhạy hơn với sai lệch lớn.
- **R²:** so sánh sai số bình phương với mức biến thiên của điểm thực tế
  quanh trung bình trên tập đang đánh giá; không phải tỷ lệ dự đoán đúng.

MAE và RMSE có đơn vị là điểm benchmark, càng nhỏ càng tốt trên cùng tập đánh giá.
R² có thể âm.

### So sánh với mô hình cơ sở

`DummyRegressor(strategy="mean")` luôn dự đoán bằng điểm trung bình của **train**.
So sánh Gaming và mô hình cơ sở trên **cùng tập test** để xem việc sử dụng
điểm CPU/GPU có giúp giảm sai số hay không.

**Bảng chỉ số, số mẫu và số nhóm ở phần này dùng để bổ sung mục đánh giá Gaming
trong báo cáo.** Các bảng xuất giữ số đầy đủ; làm tròn chỉ để hiển thị.

Đây là đánh giá lại trên bộ dữ liệu đã dùng trong quá trình phát triển,
chưa phải kiểm chứng bằng benchmark mới thu thập độc lập. Nếu tiếp tục chọn
mô hình dựa vào kết quả test này, cần ghi nhận test đã tham gia phát triển.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

eval_train_pred = eval_model.predict(eval_train[eval_features])
eval_test_pred = eval_model.predict(eval_test[eval_features])
if not np.isfinite(eval_test_pred).all() or (eval_test_pred <= 0).any():
    raise ValueError("Mô hình đánh giá tạo điểm test không hợp lệ.")

eval_baseline = DummyRegressor(strategy="mean")
eval_baseline.fit(eval_train[eval_features], eval_train["Score"])
eval_baseline_pred = eval_baseline.predict(eval_test[eval_features])


def eval_metric_row(label, actual, predicted):
    return {
        "Phạm vi đánh giá": label,
        "Số mẫu": len(actual),
        "MAE": float(mean_absolute_error(actual, predicted)),
        "RMSE": float(np.sqrt(mean_squared_error(actual, predicted))),
        "R²": float(r2_score(actual, predicted)),
    }


eval_metrics = pd.DataFrame([
    eval_metric_row("Gaming — tập train", eval_train["Score"], eval_train_pred),
    eval_metric_row("Gaming — tập test", eval_test["Score"], eval_test_pred),
    eval_metric_row("Trung bình train — đánh giá trên test",
                    eval_test["Score"], eval_baseline_pred),
])
print("KẾT QUẢ ĐÁNH GIÁ — MAE và RMSE có đơn vị điểm Time Spy Overall")
display(eval_metrics.round(4))

eval_test_predictions = eval_test[
    ["RunId", "CpuModel", "GpuModel", "GpuScore", "CpuMultiScore", "Score"]
].copy()
eval_test_predictions["CpuGpuGroup"] = eval_groups.iloc[eval_test_idx]
eval_test_predictions["Predicted"] = eval_test_pred
eval_test_predictions["BaselinePredicted"] = eval_baseline_pred
eval_test_predictions["AbsoluteError"] = np.abs(
    eval_test_predictions["Score"] - eval_test_pred
)

eval_split_manifest = data[["RunId"]].copy()
eval_split_manifest["CpuGpuGroup"] = eval_groups
eval_split_manifest["Split"] = np.where(
    eval_groups.isin(eval_test_groups), "test", "train"
)

print("Các mẫu test có sai số lớn nhất:")
display(eval_test_predictions.nlargest(10, "AbsoluteError").round(2))

eval_info = {
    "scope": "Retrospective grouped holdout evaluation on existing data",
    "new_external_validation": False,
    "method": "GroupShuffleSplit",
    "group_definition": "Normalized CpuKey + GpuKey; not physical machine ID",
    "random_state": EVAL_RANDOM_STATE,
    "test_group_fraction": EVAL_TEST_GROUP_FRACTION,
    "train_row_count": len(eval_train),
    "test_row_count": len(eval_test),
    "train_group_count": len(eval_train_groups),
    "test_group_count": len(eval_test_groups),
    "group_overlap_count": len(eval_train_groups & eval_test_groups),
    "feature_order": eval_features,
    "target": "3DMark Time Spy Overall",
    "source_excel_name": excel_name,
    "source_excel_sha256": excel_sha256,
    "evaluation_model_coefficients": eval_coefficients,
    "baseline_train_mean": float(eval_train["Score"].mean()),
    "metrics": eval_metrics.to_dict(orient="records"),
    "metrics_file": "evaluation_metrics.csv",
    "predictions_file": "evaluation_test_predictions.csv",
    "split_file": "evaluation_split.csv",
}

## 5. Huấn luyện và kiểm tra mô hình triển khai

### 5.1. Huấn luyện trên toàn bộ dữ liệu hợp lệ

Sau khi đánh giá, tạo **final_model** và huấn luyện trên toàn bộ dữ liệu hợp lệ
để xuất cho ứng dụng. Giữ nguyên thuật toán log–log và ràng buộc hệ số.

Mô hình này khác `eval_model`: số mẫu học hệ số lớn hơn, nên hệ số có thể khác.
Không lấy mô hình đã học toàn bộ dữ liệu để tính lại chỉ số rồi gọi là kết quả test.
Chỉ số phần 4 thuộc mô hình học trên tập train, không trực tiếp kiểm chứng
bộ hệ số sau khi huấn luyện lại trên toàn bộ dữ liệu.

Log/exp được đóng gói trong mô hình; truyền điểm gốc vào `predict()`.
Sau khi học, kiểm tra công thức $A\times G^b\times C^c$ khớp với `predict()`.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor

# Thứ tự đầu vào được dùng thống nhất khi train và dự đoán.
FEATURES = ["GpuScore", "CpuMultiScore"]
TARGET = "Score"

X = data[FEATURES].copy()
y = data[TARGET].copy()

# Lấy log đầu vào, sau đó học hồi quy có hệ số không âm.
input_pipeline = Pipeline([
    ("log_input", FunctionTransformer(np.log, validate=True)),
    ("linear", LinearRegression(positive=True))
])

# Tự lấy log điểm đích khi train và exp kết quả khi predict.
final_model = TransformedTargetRegressor(
    regressor=input_pipeline,
    func=np.log,
    inverse_func=np.exp
)

final_model.fit(X, y)

# Lấy hệ số từ mô hình đã fit.
linear_model = final_model.regressor_.named_steps["linear"]

a = float(linear_model.intercept_)
b = float(linear_model.coef_[0])  # Hệ số log GPU
c = float(linear_model.coef_[1])  # Hệ số log CPU
A = float(np.exp(a))

if not np.isfinite([a, A, b, c]).all() or A <= 0:
    raise ValueError("Hệ số mô hình không hợp lệ.")

if b < 0 or c < 0:
    raise ValueError("Hệ số không đáp ứng ràng buộc không âm.")

print("Đã huấn luyện trên", len(data), "dòng.")
print("Thứ tự đầu vào:", FEATURES)

display(pd.DataFrame({
    "Hệ số": ["a — hệ số chặn trên thang log", "A = exp(a)",
              "b — số mũ GPU", "c — số mũ CPU"],
    "Giá trị": [a, A, b, c]
}))

print("\nCông thức trên thang điểm gốc:")
print(
    f"Score = {A:.12g}"
    f" × GpuScore^{b:.12g}"
    f" × CpuMultiScore^{c:.12g}"
)

# Kiểm tra công thức xuất cho backend khớp predict().
pred_model = final_model.predict(X)
pred_formula = (
    A
    * X["GpuScore"].to_numpy() ** b
    * X["CpuMultiScore"].to_numpy() ** c
)

if not np.isfinite(pred_model).all() or (pred_model <= 0).any():
    raise ValueError("Mô hình tạo ra điểm dự đoán không hợp lệ.")

np.testing.assert_allclose(
    pred_model,
    pred_formula,
    rtol=1e-10,
    atol=1e-8
)

print("\nĐạt: công thức tính trực tiếp khớp với model.predict().")
print("Đạt: tăng từng điểm đầu vào không làm giảm điểm dự đoán.")

### 5.2. Kiểm tra mô hình triển khai và công thức backend

Tính MAE/RMSE/R² trên dữ liệu đã dùng để học `final_model` và xem các mẫu lệch lớn.
Đây là **chỉ số huấn luyện**, không thay thế bảng đánh giá test ở phần 4.

Tạo các tổ hợp CPU–GPU để kiểm tra điểm không giảm khi tăng từng đầu vào
và giữ đầu vào còn lại cố định. Các tổ hợp này chỉ kiểm tra công thức,
không phải benchmark thực đo và không phải tập test đánh giá độ chính xác.

Tính không giảm không bảo đảm mô hình mô tả đầy đủ hiện tượng nghẽn hiệu năng.

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# 1. Dự đoán trên dữ liệu đã dùng để train.
training_predictions = data.copy()
training_predictions["Predicted"] = final_model.predict(data[FEATURES])

training_predictions["Error"] = (
    training_predictions["Predicted"] - training_predictions["Score"]
)
training_predictions["AbsoluteError"] = (
    training_predictions["Error"].abs()
)
training_predictions["AbsolutePercentageError"] = (
    100
    * training_predictions["AbsoluteError"]
    / training_predictions["Score"]
)

training_metrics = {
    "MAE": float(mean_absolute_error(
        data["Score"], training_predictions["Predicted"]
    )),
    "RMSE": float(np.sqrt(mean_squared_error(
        data["Score"], training_predictions["Predicted"]
    ))),
    "R2": float(r2_score(
        data["Score"], training_predictions["Predicted"]
    ))
}

print("Chỉ số trên tập HUẤN LUYỆN, không phải test:")
display(pd.DataFrame([training_metrics]))

print("\n10 dòng có sai số tuyệt đối lớn nhất:")
display(
    training_predictions.nlargest(10, "AbsoluteError")[
        ["RunId", "CpuModel", "GpuModel", "Score", "Predicted",
         "AbsoluteError", "AbsolutePercentageError"]
    ].round(2)
)


# 2. Tạo các tổ hợp CPU/GPU để kiểm tra công thức.
cpu_catalog = (
    data[["CpuKey", "CpuModel", "CpuMultiScore"]]
    .drop_duplicates("CpuKey")
    .sort_values("CpuMultiScore")
)

gpu_catalog = (
    data[["GpuKey", "GpuModel", "GpuScore"]]
    .drop_duplicates("GpuKey")
    .sort_values("GpuScore")
)

prediction_grid = cpu_catalog.merge(gpu_catalog, how="cross")
prediction_grid["Predicted"] = final_model.predict(
    prediction_grid[FEATURES]
)

if (
    not np.isfinite(prediction_grid["Predicted"]).all()
    or prediction_grid["Predicted"].le(0).any()
):
    raise ValueError("Có dự đoán không hữu hạn hoặc không lớn hơn 0.")

# Giữ GPU cố định, tăng CPU Mark.
for gpu_key, group in prediction_grid.groupby("GpuKey"):
    ordered = group.sort_values("CpuMultiScore")
    if (np.diff(ordered["Predicted"]) < -1e-8).any():
        raise ValueError(f"Điểm giảm khi tăng CPU Mark: {gpu_key}")

# Giữ CPU cố định, tăng GPU Score.
for cpu_key, group in prediction_grid.groupby("CpuKey"):
    ordered = group.sort_values("GpuScore")
    if (np.diff(ordered["Predicted"]) < -1e-8).any():
        raise ValueError(f"Điểm giảm khi tăng GPU Score: {cpu_key}")

print(f"\nĐạt: kiểm tra {len(prediction_grid)} tổ hợp CPU–GPU.")
print("Không phát hiện điểm giảm khi tăng từng điểm đầu vào.")


# 3. Minh họa thay CPU với GPU có G3D Mark thấp nhất.
weakest_gpu_key = gpu_catalog.iloc[0]["GpuKey"]
example = prediction_grid.loc[
    prediction_grid["GpuKey"].eq(weakest_gpu_key)
].sort_values("CpuMultiScore")

print("\nMinh họa giữ GPU cố định, thay CPU:")
display(
    example[
        ["GpuModel", "CpuModel", "CpuMultiScore", "Predicted"]
    ].round(2)
)

## 6. Xuất mô hình và kết quả đánh giá

### Mô hình triển khai

- `model.joblib`, `model_info.json`: mô hình cuối và hệ số cho backend.
- `data_used.csv`, `excluded_rows.csv`, `reference.csv`: dữ liệu truy vết.
- `training_predictions.csv`: dự đoán trên tập huấn luyện của mô hình cuối.
- `backend_test_vectors.csv`: các đầu vào và kết quả để đối chiếu công thức.

### Kết quả đánh giá riêng

- `evaluation_metrics.csv`: bảng MAE, RMSE, R² để đưa vào báo cáo.
- `evaluation_test_predictions.csv`: điểm thực tế, dự đoán và sai số từng mẫu test.
- `evaluation_split.csv`: ghi chính xác mẫu và nhóm thuộc train/test.
- `evaluation_info.json`: phương án chia tập, nguồn dữ liệu và hệ số đánh giá.
- `evaluation_model.joblib`: mô hình chỉ học trên train để tái lập kết quả test;
  không thay thế `model.joblib` triển khai.

Đọc lại mô hình và các tệp đã lưu để kiểm tra trước khi đóng gói ZIP.
Notebook tạo thư mục phiên bản mới, không tự ghi đè mô hình trong dự án backend.

Backend phải dùng đúng điểm CPU/GPU, kiểm tra đầu vào hữu hạn và lớn hơn 0,
đồng thời nhận diện đầu vào ngoài phạm vi huấn luyện là ngoại suy.

In [ ]:
import json
import zipfile
import joblib
import sklearn

from pathlib import Path
from datetime import datetime, timezone

# Tạo thư mục riêng cho lần xuất này.
created_at = datetime.now(timezone.utc)
version = created_at.strftime("%Y%m%dT%H%M%S%fZ")
output_dir = Path(f"gaming_final_{version}")
output_dir.mkdir(exist_ok=False)

# Lấy lại hệ số trực tiếp từ model, không dùng số làm tròn.
fitted_linear = final_model.regressor_.named_steps["linear"]

coefficients = {
    "A": float(np.exp(fitted_linear.intercept_)),
    "b": float(fitted_linear.coef_[0]),
    "c": float(fitted_linear.coef_[1])
}

model_info = {
    "model_version": f"gaming-log-v1-{version}",
    "created_at_utc": created_at.isoformat(),
    "model_type": "Nonnegative log-log linear regression",
    "fit_scope": "All accepted eligible rows",
    "target": "3DMark Time Spy Overall",
    "feature_order": FEATURES,
    "feature_definitions": {
        "GpuScore": "PassMark G3D Mark; exact GPU and VRAM variant",
        "CpuMultiScore": "PassMark CPU Mark"
    },
    "formula": "Score = A * GpuScore**b * CpuMultiScore**c",
    "coefficients": coefficients,
    "input_requirements": {
        "finite": True,
        "strictly_positive": True,
        "missing_values": "Reject; do not replace with zero",
        "outside_training_ranges": "Flag as extrapolation"
    },
    "training_ranges": {
        feature: {
            "min": float(data[feature].min()),
            "max": float(data[feature].max())
        }
        for feature in FEATURES
    },
    "training_row_count": len(data),
    "excluded_row_count": len(excluded_rows),
    "cpu_count": int(data["CpuKey"].nunique()),
    "gpu_count": int(data["GpuKey"].nunique()),
    "source_excel_name": excel_name,
    "source_excel_sha256": excel_sha256,
    "versions": {
        "sklearn": sklearn.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "joblib": joblib.__version__
    },
    "training_metrics_only": training_metrics,
    "evaluation": {
        "independent_validation_of_final_model": False,
        "separate_evaluation": eval_info,
        "evaluation_model_file": "evaluation_model.joblib"
    },
    "limitations": [
        "Training metrics are not independent validation.",
        "Benchmark run conditions vary.",
        "Monotonic predictions do not guarantee realistic bottlenecks.",
        "Predictions represent Time Spy Overall, not per-game FPS.",
        "Input ranges do not establish coverage of every CPU-GPU pair."
    ]
}

# 1. Lưu model và thông tin đầy đủ.
joblib.dump(final_model, output_dir / "model.joblib")

info_path = output_dir / "model_info.json"
info_path.write_text(
    json.dumps(
        model_info,
        ensure_ascii=False,
        indent=2,
        allow_nan=False
    ),
    encoding="utf-8"
)

# 2. Lưu dữ liệu và kết quả chẩn đoán, không làm tròn.
data.to_csv(
    output_dir / "data_used.csv", index=False, encoding="utf-8-sig"
)
excluded_rows.to_csv(
    output_dir / "excluded_rows.csv", index=False, encoding="utf-8-sig"
)
reference.to_csv(
    output_dir / "reference.csv", index=False, encoding="utf-8-sig"
)
training_predictions.to_csv(
    output_dir / "training_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

# Các tổ hợp này dùng kiểm tra phép tính, không phải dữ liệu thực đo.
backend_test_vectors = prediction_grid[
    ["CpuModel", "GpuModel"] + FEATURES + ["Predicted"]
].rename(columns={"Predicted": "ExpectedScore"})

backend_test_vectors.to_csv(
    output_dir / "backend_test_vectors.csv",
    index=False,
    encoding="utf-8-sig"
)

# 3. Đọc lại model, JSON và bảng kiểm tra đã lưu.
loaded_model = joblib.load(output_dir / "model.joblib")
loaded_info = json.loads(info_path.read_text(encoding="utf-8"))
loaded_vectors = pd.read_csv(output_dir / "backend_test_vectors.csv")

expected = loaded_vectors["ExpectedScore"].to_numpy()
reloaded_predictions = loaded_model.predict(
    loaded_vectors[loaded_info["feature_order"]]
)

k = loaded_info["coefficients"]
formula_predictions = (
    k["A"]
    * loaded_vectors["GpuScore"].to_numpy() ** k["b"]
    * loaded_vectors["CpuMultiScore"].to_numpy() ** k["c"]
)

np.testing.assert_allclose(
    reloaded_predictions, expected, rtol=1e-10, atol=1e-8
)
np.testing.assert_allclose(
    formula_predictions, expected, rtol=1e-10, atol=1e-8
)

print("Đạt: model đọc lại và hệ số JSON khớp toàn bộ mẫu kiểm tra.")


# Lưu mô hình đánh giá và kết quả riêng, không ghi đè mô hình triển khai.
joblib.dump(eval_model, output_dir / "evaluation_model.joblib")
(output_dir / "evaluation_info.json").write_text(
    json.dumps(eval_info, ensure_ascii=False, indent=2, allow_nan=False),
    encoding="utf-8",
)
eval_metrics.to_csv(output_dir / "evaluation_metrics.csv", index=False, encoding="utf-8-sig")
eval_test_predictions.to_csv(
    output_dir / "evaluation_test_predictions.csv", index=False, encoding="utf-8-sig"
)
eval_split_manifest.to_csv(
    output_dir / "evaluation_split.csv", index=False, encoding="utf-8-sig"
)
loaded_eval_model = joblib.load(output_dir / "evaluation_model.joblib")
loaded_eval_predictions = pd.read_csv(output_dir / "evaluation_test_predictions.csv")
np.testing.assert_allclose(
    loaded_eval_model.predict(loaded_eval_predictions[eval_features]),
    loaded_eval_predictions["Predicted"].to_numpy(),
    rtol=1e-10, atol=1e-8,
)
print("Đạt: mô hình đánh giá đọc lại khớp dự đoán test đã lưu.")

# 4. Đóng gói và tải xuống.
export_names = [
    "model.joblib",
    "model_info.json",
    "data_used.csv",
    "excluded_rows.csv",
    "reference.csv",
    "training_predictions.csv",
    "backend_test_vectors.csv",
    "evaluation_metrics.csv",
    "evaluation_test_predictions.csv",
    "evaluation_split.csv",
    "evaluation_info.json",
    "evaluation_model.joblib"
]

zip_path = output_dir.with_suffix(".zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for name in export_names:
        archive.write(output_dir / name, arcname=name)

print("Phiên bản:", model_info["model_version"])
print("Số dòng đã train:", len(data))
print("Gói xuất:", zip_path.name)

files.download(str(zip_path))